In [1]:
from pathlib import Path
import sys

import dotenv
import pendulum
from sqlalchemy import asc, cast
from sqlalchemy import create_engine, select, BigInteger
from sqlalchemy.orm import sessionmaker
from sqlalchemy import create_engine, select, BigInteger

# Ensure repo root is on sys.path when running this file directly.
from api.config import Settings
from api.models import MessageSql
import matplotlib.pyplot as plt

house_alias = "oak"
message_type = "snapshot.spaceheat"
# start_ms = pendulum.datetime(2026, 4, 2, 0, 0, 0, tz='America/New_York').timestamp()*1000
# end_ms = pendulum.datetime(2026, 4, 4, 0, 0, 0, tz='America/New_York').timestamp()*1000
start_ms = pendulum.datetime(2026, 6, 16, 8, 0, 0, tz='America/New_York').timestamp()*1000
end_ms = pendulum.datetime(2026, 6, 16, 9, 0, 0, tz='America/New_York').timestamp()*1000

stmt = select(MessageSql).filter(
    MessageSql.message_type_name == message_type,
    MessageSql.from_alias == f"hw1.isone.me.versant.keene.{house_alias}.scada",
    MessageSql.message_created_ms <= cast(int(end_ms), BigInteger),
    MessageSql.message_created_ms >= cast(int(start_ms), BigInteger),
).order_by(asc(MessageSql.message_persisted_ms))

settings = Settings(_env_file=dotenv.find_dotenv())
engine = create_engine(settings.journaldb_url.get_secret_value())
Session = sessionmaker(bind=engine)
session = Session()
result = session.execute(stmt)
messages = result.scalars().all()

print(f"Found {len(messages)} messages")

import pickle

with open('messages.pkl', 'wb') as f:
    pickle.dump(messages, f)

print(f"Saved {len(messages)} messages to messages.pkl")

print(messages[0].payload.keys())

Found 21 messages
Saved 21 messages to messages.pkl
dict_keys(['Version', 'TypeName', 'FromGNodeAlias', 'LatestStateList', 'LatestReadingList', 'SnapshotTimeUnixMs', 'FromGNodeInstanceId'])


In [2]:
m = messages[0]
reports = [m]
relays = {}
for snapshot in reports:
    # print(snapshot.payload)
    # print(snapshot.payload.keys())

    zones = {}
    for reading in snapshot.payload['LatestReadingList']:
        if 'zone' in reading['ChannelName'] and 'whitewire' in reading['ChannelName']:
            zone_channel_name = reading['ChannelName']
            if zone_channel_name not in zones:
                zones[zone_channel_name] = {'timestamps': [], 'values': []}
            zone_value = reading['Value']
            zone_value_timestamp = int(reading['ScadaReadTimeUnixMs']/1000)
            print(f"{zone_channel_name}: {zone_value} at {zone_value_timestamp}")
            zones[zone_channel_name]['timestamps'].append(zone_value_timestamp)
            zones[zone_channel_name]['values'].append(zone_value)
    
    for state in snapshot.payload['LatestStateList']:
        if 'StateEnum' not in state:
            continue
        if state['StateEnum'] == 'top.state':
            top_state = state['State']
        if state['StateEnum'] == 'gw1.main.auto.state':
            if top_state=='Auto':
                print(state['State'])
            else:
                print(top_state)

zone1-living-rm-whitewire-pwr: 51 at 1781611198
zone2-garage-whitewire-pwr: 0 at 1781611130
zone3-gear-rm-whitewire-pwr: 0 at 1781611187
zone4-upstairs-whitewire-pwr: 0 at 1781611176
Admin
